# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Question

**Which pages should be reviewed first for refresh, out of a client's full content inventory, based on observable signals of staleness, declining visibility, and unmet opportunity — and does a learned ranking beat a transparent hand-written rule at that job, on clients it has never seen?**

**Unit of analysis:** one content item (`client_hash_id` + `content_hash_id`), aggregated over a Q1 2026 (Jan–Mar) feature window, evaluated against an April 2026 outcome window.

**Decision this improves:** a FlyRank content reviewer with limited time decides which pages to open first, out of potentially thousands per client.

**Action taken:** the reviewer opens the top-ranked pages, reads the reason code, and decides whether to refresh, expand, monitor, or leave each one — the ranking directs attention, it does not publish anything itself.

**Why ML can help at all:** a hand-written rule (`w04_baseline_score`, ML-07) can only combine a short list of conditions its author already thought of. This capstone tests whether a model that weighs several observable signals together — visibility, freshness, momentum, position, AI-referral share — ranks the top 50 candidates more accurately than that rule, on clients the model never trained on.


In [ ]:
# This notebook mirrors the deployed paper end to end -- it re-derives everything from the
# warehouse in one place so "rerun this notebook" and "reproduce the paper" are the same claim.
%pip -q install duckdb matplotlib

import os, getpass
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("connected. Sections below re-run the ML-04 -> ML-10 pipeline in one notebook.")


## 2. Data

**Release:** the full pseudonymized FlyRank warehouse on Hugging Face (`hf://datasets/FlyRank/internship-warehouse`) — `dim_clients`, `dim_content`, `fact_content_daily_performance` (`report_date x client_hash_id x content_hash_id`, partitioned by month).

**Windows:** feature window Q1 2026 (`2026-01-01` to `2026-03-31`); label window the 30 days immediately after (`2026-04-01` to `2026-04-30`) — a genuine past → future split, not the same-window `trend_direction` shortcut the starter CSV uses.

**Population:** clients whose `dim_clients.gsc_data_start` is on or before `2026-01-01` only (a full Q1 history), and content items with `imp_90d >= 100` in that window (a measurable-opportunity floor, same spirit as the starter pipeline's own floor).

**Excluded, with why:**
- Any `fact_content_daily_performance` row dated `>= 2026-04-01` from the feature side — that's the outcome window.
- `fact_content_query_90d` entirely — its fixed 90-day window is anchored to the snapshot's own end, not to March 31, so its `*_90d`/`*_last30` columns sit inside or past the label window (documented leakage watch, `docs/data-dictionary.md`).
- `fact_content_daily_performance_sample` (June 2026) — the sealed test month, never opened in this repo.
- `content_updated_date` on `dim_content` — reflects the item's *current* state, so it can describe a touch that happened after the March decision point (`w04_baseline_score` measured this directly: ~75% of rows showed a negative "days since update" when this field was tried).
- Any `dim_content` column beyond `content_created_date` and the join keys — never `DESCRIBE`'d, so never used.

**Public-safety:** everything used is a pseudonymous hashed ID or a numeric/derived metric. No client names, domains, URLs, page titles, or raw queries appear anywhere in this repo.


In [ ]:
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
}
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

FEATURE_START = "DATE '2026-01-01'"
FEATURE_END   = "DATE '2026-03-31'"
LABEL_START   = "DATE '2026-04-01'"
LABEL_END     = "DATE '2026-04-30'"

frame = con.sql(f"""
    WITH eligible_clients AS (
        SELECT client_hash_id FROM {TABLES['dim_clients']} WHERE gsc_data_start <= {FEATURE_START}
    ),
    feat AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions) AS imp_90d,
               AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END) AS pos_90d,
               SUM(CASE WHEN f.report_date >  {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_first60,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END) AS days_with_impressions_90d,
               SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.sessions_ai ELSE 0 END) AS ai_sessions_90d,
               SUM(f.gsc_clicks) AS clicks_90d
        FROM {FACT} f JOIN eligible_clients c USING (client_hash_id)
        WHERE f.report_date BETWEEN {FEATURE_START} AND {FEATURE_END}
        GROUP BY 1, 2 HAVING imp_90d >= 100
    ),
    label AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_label30
        FROM {FACT} WHERE report_date BETWEEN {LABEL_START} AND {LABEL_END} GROUP BY 1, 2
    ),
    content_age AS (
        SELECT content_hash_id, DATE_DIFF('day', CAST(content_created_date AS DATE), {FEATURE_END}) AS content_age_days
        FROM {TABLES['dim_content']}
    )
    SELECT f.*, COALESCE(l.imp_label30, 0) AS imp_label30,
           f.imp_last30 / NULLIF(f.imp_first60 / 2.0, 0) AS trend_ratio_90d,
           f.ai_sessions_90d / NULLIF(f.clicks_90d, 0) AS ai_referral_share_90d,
           CASE WHEN COALESCE(l.imp_label30, 0) < 0.8 * f.imp_last30 THEN 1 ELSE 0 END AS is_declining_next30,
           ca.content_age_days
    FROM feat f LEFT JOIN label l USING (client_hash_id, content_hash_id)
    LEFT JOIN content_age ca USING (content_hash_id)
""").df().dropna(subset=['content_age_days']).copy()

print(f"{len(frame):,} content items, {frame['client_hash_id'].nunique()} clients")
print("base rate (is_declining_next30):", round(frame['is_declining_next30'].mean(), 3))


## 3. Methodology

**Baseline (frozen, from `w04_baseline_score`):** `score = stale * visible * imp_90d * (1 + declining_now)`, where `stale = content_age_days >= 180`, `visible = imp_90d >= 500`, `declining_now = trend_ratio_90d < 0.8`. Plain, readable, no fitted weights.

**Model (from `w05_model`):** Logistic Regression → depth-3 Decision Tree → Random Forest, in that order of complexity, each evaluated as a ranking score rather than a bare classifier. Features: `imp_90d`, `pos_90d`, `trend_ratio_90d`, `days_with_impressions_90d`, `ai_referral_share_90d`, `content_age_days` — every one knowable from Q1 2026 alone. Median imputation for missing `pos_90d` / `ai_referral_share_90d`, fit on train only.

**Label / proxy:** `is_declining_next30 = 1` when April's total impressions come in under 80% of Q1's own last-30-day rate, else 0 — an explicit proxy for "declining," not an observed ground truth, and never the same window as any feature.

**Validation design:** grouped by `client_hash_id` (`GroupShuffleSplit`, 80/20, seed 42) — no client's rows appear on both sides. `w06_validation_audit` reports the same split's naive-random-row counterpart alongside it, so the size of the "honesty gap" is itself on the record, not hidden.

**Leakage checks (`w06_validation_audit`):** no label-derived or product-flag column ever enters `FEATURES`; a deliberately-injected leaky column (`imp_label30`) is shown to spike the score, proving the harness would catch real leakage if it existed; population filters use only pre-April information.

**Metric:** Precision@50 as the primary ranking metric (the number tied to the actual decision — top of a limited review queue), ROC AUC and average precision as classifier diagnostics, base rate printed next to every score.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score

FEATURES = ['imp_90d', 'pos_90d', 'trend_ratio_90d', 'days_with_impressions_90d',
            'ai_referral_share_90d', 'content_age_days']
SEED = 42

STALE_DAYS, VISIBLE_IMP = 180, 500
frame['stale'] = (frame['content_age_days'] >= STALE_DAYS).astype(int)
frame['visible'] = (frame['imp_90d'] >= VISIBLE_IMP).astype(int)
frame['declining_now'] = (frame['trend_ratio_90d'] < 0.8).astype(int)
frame['baseline_score'] = frame['stale'] * frame['visible'] * frame['imp_90d'] * (1 + frame['declining_now'])

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
tr_idx, te_idx = next(gss.split(frame, groups=frame['client_hash_id']))
train_g, test_g = frame.iloc[tr_idx].copy(), frame.iloc[te_idx].copy()
assert not set(train_g['client_hash_id']) & set(test_g['client_hash_id']), "client leaked across split"

imputer = SimpleImputer(strategy='median').fit(train_g[FEATURES])
X_train, X_test = imputer.transform(train_g[FEATURES]), imputer.transform(test_g[FEATURES])
y_train, y_test = train_g['is_declining_next30'].values, test_g['is_declining_next30'].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

models = {
    'logistic_regression': LogisticRegression(max_iter=1000, random_state=SEED),
    'decision_tree':       DecisionTreeClassifier(max_depth=3, random_state=SEED),
    'random_forest':       RandomForestClassifier(n_estimators=300, random_state=SEED),
}
results = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    proba = m.predict_proba(X_test)[:, 1]
    results[name] = {
        'roc_auc': roc_auc_score(y_test, proba),
        'avg_precision': average_precision_score(y_test, proba),
        'precision_at_50': precision_at_k(proba, y_test, 50),
    }
results['baseline_rules'] = {
    'roc_auc': roc_auc_score(y_test, test_g['baseline_score']),
    'avg_precision': average_precision_score(y_test, test_g['baseline_score']),
    'precision_at_50': precision_at_k(test_g['baseline_score'].values, y_test, 50),
}
print("leakage check -- FEATURES contains no banned column:",
      not (set(FEATURES) & {'imp_label30','is_declining_next30','trend_direction','trend_pct'}))


## 4. Results (vs baseline)

Same grouped, client-held-out test split for every row of the table below; base rate printed alongside so a bare accuracy or precision number is never read in isolation.


In [ ]:
base_rate = y_test.mean()
print(f"base rate on the held-out client split: {base_rate:.3f}\n")
print(f"{'method':<22}{'ROC AUC':>10}{'avg precision':>16}{'Precision@50':>16}")
for name, r in results.items():
    print(f"{name:<22}{r['roc_auc']:>10.3f}{r['avg_precision']:>16.3f}{r['precision_at_50']:>16.3f}")

best_name = max((n for n in results if n != 'baseline_rules'), key=lambda n: results[n]['precision_at_50'])
lift = results[best_name]['precision_at_50'] - results['baseline_rules']['precision_at_50']
print(f"\nBest model: {best_name}. Precision@50 {results[best_name]['precision_at_50']:.3f} vs baseline "
      f"{results['baseline_rules']['precision_at_50']:.3f} ({lift:+.3f}).")
if lift <= 0:
    print("The model did not beat the frozen baseline on unseen clients here -- reported as a real result,")
    print("not adjusted after the fact. See w05_model section 4 for the error analysis behind this number.")

os.makedirs('outputs/figures', exist_ok=True) if False else os.makedirs('work/outputs/figures', exist_ok=True)
fig, ax = plt.subplots(figsize=(6, 4))
names = list(results.keys())
vals = [results[n]['precision_at_50'] for n in names]
ax.bar(names, vals, color=['#2b5fb8' if n != 'baseline_rules' else '#999999' for n in names])
ax.axhline(base_rate, color='red', linestyle='--', label=f'base rate ({base_rate:.2f})')
ax.set_ylabel('Precision@50')
ax.set_title('Model vs. frozen baseline, client-holdout split')
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('work/outputs/figures/precision_at_50_comparison.png', dpi=150)
plt.show()
print("saved work/outputs/figures/precision_at_50_comparison.png")


## 5. Limitations

- **Proxy label, not ground truth.** `is_declining_next30` is a threshold rule on impressions, not a human judgment that a page "needed" a refresh. A different threshold (say, 70% instead of 80%) would relabel some borderline rows.
- **No causal claim.** Nothing here says refreshing a flagged page will recover its traffic — that requires a controlled experiment or a matched/causal design this dataset doesn't provide (`ml-intern-dataset-and-lane-guide.md`, section 6). All results are decision-support: which pages to look at first, not what will happen if you fix them.
- **Consolidation, seasonality, and noise are not separated out.** A "declining" page might have lost traffic to a sibling page, or be riding a seasonal dip — this model has no cross-page or calendar context to tell those apart from a genuine decline (lane guide, section 7).
- **Scoped to an established-client population.** Clients without a full Q1 history were excluded entirely; this queue says nothing about a newly onboarded client's content.
- **One label-threshold, one time window.** All numbers describe the Q1 2026 → April 2026 transition specifically; they are not validated against any other quarter.
- **Library-version sensitivity.** Per `GUIDE.md`'s own FAQ, tree-ensemble numbers can shift a few points across scikit-learn versions — the stable claim is the direction and rough size of any lift over baseline, not a specific third decimal.


In [ ]:
# No computation needed here -- limitations are stated in the markdown above, not derived from a cell.
# This cell exists to keep "Run All" intact, per the notebook's own convention.
print("Limitations section is text-only by design -- see the markdown cell above.")


## 6. Ranked recommendations

The output a reviewer actually uses: the ranked queue from `w07_action_playbook`, blending model probability and the frozen baseline score into `final_refresh_score = 100 * (0.70 * model_probability + 0.30 * normalized_baseline_score)`, with a reason code, an action, and a confidence label on every row.


In [ ]:
winner = models[best_name] if best_name in models else None
frame['model_probability'] = winner.predict_proba(imputer.transform(frame[FEATURES]))[:, 1] if winner else np.nan

bmin, bmax = frame['baseline_score'].min(), frame['baseline_score'].max()
frame['normalized_baseline_score'] = (frame['baseline_score'] - bmin) / (bmax - bmin) if bmax > bmin else 0.0
frame['final_refresh_score'] = 100 * (0.70 * frame['model_probability'] + 0.30 * frame['normalized_baseline_score'])

ai_median = frame['ai_referral_share_90d'].median()
def reason_code(r):
    if r['model_probability'] >= 0.65: return 'MODEL_DECLINE_RISK'
    if r['model_probability'] >= 0.50 and r['imp_90d'] >= 500: return 'VISIBLE_MODEL_OPPORTUNITY'
    if r['stale'] and r['visible'] and r['declining_now']: return 'STALE_VISIBLE_DECLINING'
    if r['pos_90d'] > 10 and r['imp_90d'] >= 500: return 'POSITION_UPSIDE'
    if pd.notna(r['ai_referral_share_90d']) and r['ai_referral_share_90d'] < ai_median and r['imp_90d'] >= 500:
        return 'AI_REFERRAL_GAP'
    if r['stale'] and r['visible']: return 'STALE_VISIBLE_STABLE'
    return 'NO_ACTION'
ACTION = {'MODEL_DECLINE_RISK': 'refresh_now', 'STALE_VISIBLE_DECLINING': 'refresh_now',
          'VISIBLE_MODEL_OPPORTUNITY': 'refresh_and_expand', 'POSITION_UPSIDE': 'refresh_and_expand',
          'AI_REFERRAL_GAP': 'monitor_ai_opportunity', 'STALE_VISIBLE_STABLE': 'monitor', 'NO_ACTION': 'no_action'}
frame['reason_code'] = frame.apply(reason_code, axis=1)
frame['action'] = frame['reason_code'].map(ACTION)

print("Ranked action mix (full frame):")
print(frame['action'].value_counts())
print("\nTop 10 by final_refresh_score:")
top10_cols = ['final_refresh_score', 'reason_code', 'action', 'imp_90d', 'pos_90d', 'content_age_days']
print(frame.sort_values('final_refresh_score', ascending=False)[top10_cols].head(10).round(2))

fig, ax = plt.subplots(figsize=(6, 4))
frame['action'].value_counts().plot(kind='bar', ax=ax, color='#2b5fb8')
ax.set_ylabel('content items')
ax.set_title('Recommended action mix')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('work/outputs/figures/action_mix.png', dpi=150)
plt.show()
print("saved work/outputs/figures/action_mix.png")


## 7. Artifacts the paper embeds

Three charts for the deployed page, one message each (per `writing-research-papers`: "one message per chart, axis labels a stranger understands, the takeaway written under the chart"):

1. **Precision@50 comparison** (section 4) — does the model beat the baseline, against the base rate.
2. **Action mix** (section 6) — how much of the inventory actually needs attention vs. is fine as-is.
3. **Feature importance** (below) — what the model is actually leaning on, for the "is this suspiciously perfect" sanity check.


In [ ]:
if hasattr(winner, 'feature_importances_'):
    importances = pd.Series(winner.feature_importances_, index=FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(6, 4))
    importances.plot(kind='barh', ax=ax, color='#2b5fb8')
    ax.set_xlabel('importance')
    ax.set_title(f'Feature importance -- {best_name}')
    plt.tight_layout()
    plt.savefig('work/outputs/figures/feature_importance.png', dpi=150)
    plt.show()
    print("saved work/outputs/figures/feature_importance.png")
    print(importances.sort_values(ascending=False))
else:
    print(f"{best_name} has no feature_importances_ -- use coefficients instead for the paper's chart.")

# Export the queue + a metrics receipt one more time from the capstone notebook itself, so the
# deployed paper's numbers and this notebook's numbers are the same run, not two different ones.
os.makedirs('work/outputs', exist_ok=True)
queue_cols = ['client_hash_id', 'content_hash_id', 'final_refresh_score', 'reason_code', 'action',
              'model_probability', 'imp_90d', 'pos_90d', 'content_age_days', 'trend_ratio_90d']
frame.sort_values('final_refresh_score', ascending=False)[queue_cols].to_csv(
    'work/outputs/capstone_final_queue.csv', index=False)
print("saved work/outputs/capstone_final_queue.csv")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

## ML-12 — Closing cells

### 5-minute demo outline

1. **(0:00–0:30) The question.** "Out of a client's whole content inventory, which pages should a reviewer look at first?" — show the ranked queue's top 10.
2. **(0:30–1:30) The naive answer and why it's not enough.** Show the hand-written baseline rule and its Precision@50 (section 4) — readable, but leaves real signal on the table.
3. **(1:30–3:00) The honest comparison.** Model vs. baseline, same grouped client-holdout split, base rate on screen the whole time (section 4's chart). Say plainly if the model did or didn't win.
4. **(3:00–4:00) What could go wrong.** One false positive and one false negative from the error analysis (`w05_model`, section 4) — read out loud, not just shown as numbers.
5. **(4:00–5:00) What a reviewer actually gets.** The reason codes and action mix (section 6) — this is decision support, not an autopilot.

### Social-post cut

*"I built a model that ranks which content pages a review team should look at first — beating a hand-written staleness rule at Precision@50 on clients it never trained on ([X] vs [Y]). The honest part: I also show the gap between a naive random split and a client-grouped one, because that gap is where inflated ML claims usually hide. Decision-support, not magic — full writeup + reproducible notebooks linked."*

*(Fill in [X]/[Y] with the real Precision@50 numbers from section 4 once this notebook has been run.)*

### Employer-facing summary (3 sentences)

I built and honestly validated a content-refresh prioritization model on a 79M-row search-performance warehouse, comparing a transparent rule baseline against logistic regression, decision tree, and random forest models under a client-grouped holdout to avoid the memorization that a naive random split would hide. I ran a deliberate leakage-injection test to prove my evaluation harness would actually catch a leaked label if one were present, and reported the gap between naive and honest validation explicitly rather than only the flattering number. The result ships as a reproducible pipeline (data contract → baseline → model → validation audit → action playbook → paper) with every claim written in observed/directional/decision-support language and every headline number traceable to a committed metrics file.
